In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your project
import os
os.chdir('/content/drive/MyDrive/nanoGPT')  # adjust path to where you uploaded it

# Install dependencies
!pip install datasets tiktoken wandb tqdm torch

# Check GPU
!nvidia-smi


Mounted at /content/drive
Wed Apr 15 07:56:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
# Run training for baseline
!python -u train.py config/train_shakespeare_char_baseline.py \
  --device=cuda \
  --compile=False | tee logs/baseline.log

# Run training for lower model width
!python train.py config/train_shakespeare_char_model_width_128.py \
  --device=cuda \
  --compile=False | tee logs/model_width_128.log

# Run training for lower context
!python train.py config/train_shakespeare_char_large_LR.py \
  --device=cuda \
  --compile=False | tee logs/context.log

# Run training for larger LR
!python train.py config/train_shakespeare_char_context.py \
  --device=cuda \
  --compile=False | tee logs/large_LR.log

# Run training for smaller LR
!python train.py config/train_shakespeare_char_small_LR.py \
  --device=cuda \
  --compile=False | tee logs/small_LR.log

# Run training for smaller model depth
!python train.py config/train_shakespeare_char_model_depth_2.py \
  --device=cuda \
  --compile=False | tee logs/depth2.log

# Run training for larger model depth
!python train.py config/train_shakespeare_char_model_depth_8.py \
  --device=cuda \
  --compile=False | tee logs/depth8.log


Overriding config with config/train_shakespeare_char_baseline.py:
# Baseline configuration
out_dir = 'out-shakespeare-baseline'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False # Changed to True for Google colab
wandb_log = False
wandb_project = 'nanoGPT-assignment'
wandb_run_name = 'baseline'
dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2
learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100
weight_decay = 1e-1
device = 'cuda' # "mps" for mac, "cuda" for windows, if you have a GPU
compile = False # set True only on Linux with GPU
#init_from = 'resume' #resume from the save checkpoint
Overriding: device = cuda
Overriding: compile = False
tokens per iteration will be: 16,384
found vocab_size = 65 (inside data/shakespeare_char/meta.pkl)
Initializing a new model from scratch
number of parameters: 10.65M
/c

In [ ]:
import os
os.chdir('/content/drive/MyDrive/nanoGPT')

# Baseline
!python sample.py --out_dir=out-shakespeare-baseline \
--device=cuda --num_samples=3 --max_new_tokens=500 > baseline_samples.txt

# Low model width 
!python sample.py --out_dir=out-shakespeare-char_model_width_128 \
--device=cuda --num_samples=3 --max_new_tokens=500 > model-width_samples.txt

# Low context 
!python sample.py --out_dir=out-shakespeare_char_context \
--device=cuda --num_samples=3 --max_new_tokens=500 > context_samples.txt

# Larger LR
!python sample.py --out_dir=out-shakespeare_char_large_LR \
--device=cuda --num_samples=3 --max_new_tokens=500 > larger-LR_samples.txt

# Smaller LR
!python sample.py --out_dir=out-shakespeare_char_small_LR \
--device=cuda --num_samples=3 --max_new_tokens=500 > small-LR_samples.txt

# Smaller model depth
!python sample.py --out_dir=out-shakespeare_char_model_depth_2 \
--device=cuda --num_samples=3 --max_new_tokens=500 > model-depth2_samples.txt

# Larger model depth
!python sample.py --out_dir=out-shakespeare_char_model_depth_8 \
--device=cuda --num_samples=3 --max_new_tokens=500 > model-depth8_samples.txt



/content/drive/MyDrive/nanoGPT/sample.py:32: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
Traceback (most recent call last):
  File "/content/drive/MyDrive/nanoGPT/sample.py", line 38, in <module>
    checkpoint = torch.load(ckpt_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1540, in load
    return _load(
           ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 2143, in _load
    result = unpickler.load()
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_weights_only_unpickler.py", line 539, in load
    self.append(self.persistent_load(pid))
                ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py